Chama os notebooks de api para verificar se o cep solicitado já existe na base ou não

In [0]:
# CEP e numero informados pelo analista
dbutils.widgets.text("cep", "08717310", "CEP")
dbutils.widgets.text("numero", "123", "Numero")

cep = dbutils.widgets.get("cep")
numero = dbutils.widgets.get("numero")

# Executa os dois notebooks de extracao
dbutils.notebook.run("../src/01_api_viaCep", 120, {"cep": cep})
dbutils.notebook.run("../src/02_api_brasilapi", 120, {"cep": cep})

print("Consultas das APIs concluidas!")

Consulta as tabelas de cep após execução da api (função Fcn_Logradouro)

In [0]:
%sql
SELECT *
FROM geocep.default.Fcn_Logradouro(
    :cep,
    :numero
);

In [0]:
from pyspark.sql.functions import current_date, year, month

# Aproveita o resultado da célula SQL anterior
df = _sqldf

# Adiciona ano e mês para particionamento
df = (
    df.withColumn("ano", year(current_date()))
      .withColumn("mes", month(current_date()))
)

# Grava o resultado em Parquet com compressão Snappy
(
    df.write
    .mode("overwrite")
    .option("compression", "snappy")
    .partitionBy("ano", "mes")
    .parquet("/Volumes/geocep/default/geocep/output/")
)

print("Parquet gerado com sucesso!")
display(df)